In [1]:
from IPMSP_module import solve_instance, run_qaoa_with_params, is_feasible, plot_gantt_chart, invert_counts, assign_jobs,get_job_assignment,plot_bar_chart
import time
from matplotlib import rcParams

In [6]:
# 定义问题参数
n = 5
m = 3
inst_L = [3,3,2,1,1]
n_runs = 1
# optimizers = ['COBYLA', 'Nelder-Mead', 'Powell', 'BFGS', 'SLSQP']
optimizers = ['Powell']
start = time.process_time()
#判定
if m <= 0:
    raise ValueError("车间数目不能小于0")
if m > 5:
    raise ValueError("车间数目不能大于5")
if n < m:
    raise ValueError("作业数不能小于车间数目")
if len(inst_L) != n:
    raise ValueError("作业长度数目要等于作业数目")
if m * n > 32:
    raise ValueError("比特数(车间数目*作业数目)超过最大值(30)")

# 调用solve_instance函数，运行优化器进行求解
results, Q, g, c = solve_instance(n, m, inst_L, n_runs=n_runs, optimizers=optimizers)

# 查看优化结果
print("优化结果：")
for opt in optimizers:
    print(f"Optimizer: {opt}")
    for run_info in results[f"N_{n}M_{m}"][opt]:
        print(run_info)

# 简单策略：从第一个优化器中找到能量最低的一组参数
chosen_optimizer = optimizers[0]
runs = results[f"N_{n}M_{m}"][chosen_optimizer]
best_run = min(runs, key=lambda x: x['energy'])
best_beta = [best_run['beta']]
best_gamma = [best_run['gamma']]

# 使用最佳参数运行QAOA电路
avg_energy, counts, Q, g, c = run_qaoa_with_params(n, m, inst_L, best_beta, best_gamma, shots=10000)
print(f'优化后平均能量 = {avg_energy}')

classical_assignments, classical_bitstring, classical_decimal = assign_jobs(n, m, inst_L)
print("机器分配方案：")
for mm, assign_list in enumerate(classical_assignments):
    jobs_on_machine = [f"J{j_id}(len={length})" for (j_id, length) in assign_list]
    print(f"Machine {mm}: {jobs_on_machine}")
end1 = time.process_time()
print("运行时间:", end1 - start)
# print("二进制解:", classical_bitstring)
# print("十进制解:", classical_decimal)

run0
随机初始点[0.1986637  6.18219549]
optimizer=Powell
优化结果：
Optimizer: Powell
{'run': 0, 'beta': 0.27749331915970793, 'gamma': 6.18141402045929, 'energy': -97.00562969086594}
优化后平均能量 = -98.2584
机器分配方案：
Machine 0: ['J0(len=3)', 'J4(len=1)']
Machine 1: ['J1(len=3)']
Machine 2: ['J2(len=2)', 'J3(len=1)']
运行时间: 5.484375


In [7]:
sorted_dict = invert_counts(dict(sorted(counts.items(), key=lambda item: item[1], reverse=True)))
decimal_counts = {}
for item, count in sorted_dict.items():
    decimal_number = str(int(item, 2))
    decimal_counts[decimal_number] = count

keys = list(decimal_counts.keys())[:30]
values = list(decimal_counts.values())[:30]

binary_solutions = []
for dec_str in keys:
    dec_val = int(dec_str)
    bin_str = bin(dec_val)[2:].zfill(n*m)
    binary_solutions.append(bin_str)

colors = []
feasible_solution_for_gantt = None
for bin_str in binary_solutions:
    if is_feasible(bin_str, n, m):
        colors.append('red')
        if feasible_solution_for_gantt is None:
            feasible_solution_for_gantt = bin_str
    else:
        colors.append('grey')

if str(classical_decimal) not in keys:
    classical_value = list(decimal_counts.values())[0]
    keys.insert(0, str(classical_decimal))
    values.insert(0, classical_value)
    binary_solutions.insert(0, classical_bitstring)
    colors.append('red')
    if feasible_solution_for_gantt is None:
        feasible_solution_for_gantt = classical_bitstring

# 如果找到可行解，就绘制甘特图

if feasible_solution_for_gantt is not None:
    plot_bar_chart(keys,values,colors)
    plot_gantt_chart(classical_bitstring, n, m, inst_L)
else:
    print("No feasible solution found.")

end2 = time.process_time()
print("运行时间:", end2 - start)

运行时间: 5.5625
